In [1]:
# Install cuOpt (Linux + GPU = works perfectly here)
!pip install cuopt-cu12 --extra-index-url https://pypi.nvidia.com -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 MB 172.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 254.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.0/450.0 kB 99.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 562.9/562.9 MB 61.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.8/885.8 kB 250.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 228.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 MB 45.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 191.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 218.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.1/109.1 MB 98.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.7/39.7 MB 165.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 246.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
!pip install cuopt-cu12==26.2.* --extra-index-url https://pypi.nvidia.com -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 MB 28.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 138.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.4/428.4 kB 129.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.5/532.5 MB 34.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 901.3/901.3 kB 240.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 215.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.2/684.2 MB 42.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 55.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 132.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 24.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 40.3 MB/s eta 0:00:00


In [ ]:
from cuopt import routing
import cudf
import numpy as np

# ── Matrices ──────────────────────────────────────────────────────
cost_matrix = cudf.DataFrame([
    [0, 5, 4, 3, 5],    #stop 1
    [5, 0, 6, 4, 3],    #stop 2
    [4, 8, 0, 4, 2],    #stop 3
    [1, 4, 3, 0, 4],    #stop 4
    [3, 3, 5, 6, 0],    #stop 5
], dtype=np.float32)

time_matrix = cudf.DataFrame([
    [ 0, 10,  8,  6, 10],
    [10,  0, 12,  8,  6],
    [ 8, 16,  0,  8,  4],
    [ 2,  8,  6,  0,  8],
    [ 6,  6, 10, 12,  0],
], dtype=np.float32)

n_locations = len(cost_matrix)
n_vehicles = 3
n_orders = 4  # ← tell the model how many delivery tasks there are

# ── Data Model ────────────────────────────────────────────────────
data_model = routing.DataModel(n_locations, n_vehicles, n_orders)  # ← added n_orders
data_model.add_cost_matrix(cost_matrix)
data_model.add_transit_time_matrix(time_matrix)

# Vehicle time windows
data_model.set_vehicle_time_windows(
    cudf.Series([0, 0, 0]),       # earliest start
    cudf.Series([100, 100, 100])  # latest end
)

# Capacity dimension
data_model.add_capacity_dimension(
    "cargo",
    cudf.Series([30, 40, 40, 30]),  # demand per order
    cudf.Series([75, 75, 75])        # capacity per vehicle
)

# Orders
data_model.set_order_locations(cudf.Series([1, 2, 3, 4]))
data_model.set_order_time_windows(
    cudf.Series([3, 5, 1, 4]),     # earliest
    cudf.Series([20, 30, 20, 40])  # latest
)
data_model.set_order_service_times(cudf.Series([3, 1, 8, 4]))

# ── Solve ─────────────────────────────────────────────────────────
solver_settings = routing.SolverSettings()
solver_settings.set_time_limit(5)

solution = routing.Solve(data_model, solver_settings)

if solution.get_status() == 0:
    print("✅ Solution found!")

    routes = solution.get_route()
    print("\nFull route table:")
    print(routes.to_pandas().to_string())
else:
    print("❌ No solution found, status:", solution.get_status())

✅ Solution found!

Full route table:
   route  arrival_stamp  truck_id  location      type
0      0            0.0         0         0     Depot
1      0           10.0         0         1  Delivery
2      1           25.0         0         2  Delivery
3      0           34.0         0         0     Depot
4      0            0.0         1         0     Depot
5      2            6.0         1         3  Delivery
6      3           22.0         1         4  Delivery
7      0           32.0         1         0     Depot
